## しっぺ返し戦略と裏切り戦略の利得表

In [99]:
import random

class Player:
    def __init__(self):
        # プレイヤーが行う戦略更新関数
        self.strgFn = random.choice([
            self.tit_for_tat, 
            self.all_d
        ])

        # 初期行動を決める
        # しっぺ返し戦略なら最初は C、裏切り戦略なら D
        if self.strgFn == self.tit_for_tat:
            self.strg = "C"
        else:
            self.strg = "D"

        # プレイヤーの利得
        self.reset()
        

    # しっぺ返し戦略
    def tit_for_tat(self, opponent):
        self.nextStrg = opponent.strg

    # 裏切り戦略
    def all_d(self, opponent):
        self.nextStrg = "D"

    def reset(self):
        self.payoff = 0

    def get_payoff(self, p):
        self.payoff += p

    def upd_strg(self) -> None:
        self.strg = self.nextStrg


In [100]:
# 1対1のしっぺ返し戦略と裏切り戦略のシミュレーション
def simulate(players, strgMat, W, Tmax):
    for t in range(Tmax):
        # print(f"--- {t + 1}回目 ---")

        # 利得の計算
        for i, p in enumerate(players):
            # 1対1なので、自分が0番なら相手は1番、自分が1番なら相手は0番
            opponent = players[1 - i]

            # 自分の行動と相手の行動から利得を取得
            payoff = strgMat[p.strg][opponent.strg] * (W ** t)

            # 利得を加算
            p.get_payoff(payoff)

            # print(f"Player{i + 1}: {p.strg}, payoff +{payoff}")

        # 次の行動を決める
        for i, p in enumerate(players):
            # 相手プレイヤーを取得
            opponent = players[1 - i]

            # 自分の戦略関数に従って nextStrg を決める
            p.strgFn(opponent)

        # 全員の行動を同時に更新する
        for p in players:
            p.upd_strg()

strgMat = {
    "C": {"C": 3, "D": 0},
    "D": {"C": 8, "D": 1}
}

tft_1 = Player()
tft_2 = Player()
tft_1.strgFn = tft_1.tit_for_tat
tft_2.strgFn = tft_2.tit_for_tat
tft_1.strg = "C"
tft_2.strg = "C"

all_d_1 = Player()
all_d_2 = Player()
all_d_1.strgFn = all_d_1.all_d
all_d_2.strgFn = all_d_2.all_d
all_d_1.strg = "D"
all_d_2.strg = "D"

W = 0.8
Tmax = 100

# tit_for_tat vs tit_for_tat
tft_1.reset(), tft_2.reset()
simulate([tft_1, tft_2], strgMat, W, Tmax)
print(f"PayOff> TFT1: {tft_1.payoff}, TFT2: {tft_2.payoff}")
# 利得（TFT）: Σ 利得n * 割引率w**t
# 3 / (1 - 0.8) = 15

# tit_for_tat vs all_d
tft_1.reset(), all_d_1.reset()
simulate([tft_1, all_d_1], strgMat, W, Tmax)
print(f"PayOff> TFT1: {tft_1.payoff}, All-D: {all_d_1.payoff}")
# 利得（TFT）: Σ 利得n + 利得t * 割引率w**t （t=1の際に戦略をCからDにするため利得が変わる）
# 0 + 1 * 0.8 / (1 - 0.8)
# 利得（All-D）: Σ 利得n + 利得t * 割引率w**t （t=1の際に相手が戦略をCからDにするため利得が変わる）
# 8 + 1 * 0.8 / (1 - 0.8)

# all_d vs all_d
all_d_1.reset(), all_d_2.reset()
simulate([all_d_1, all_d_2], strgMat, W, Tmax)
print(f"PayOff> All-D1: {all_d_1.payoff}, All-D2: {all_d_2.payoff}")
# 利得（All-D）: Σ 利得n * 割引率w**t
# 1 / (1 - 0.8)

PayOff> TFT1: 14.999999996944451, TFT2: 14.999999996944451
PayOff> TFT1: 3.999999998981482, All-D: 11.999999998981485
PayOff> All-D1: 4.999999998981484, All-D2: 4.999999998981484


## 数式メモ

初回を $t=0$、割引率を $W^t$、繰り返し回数を $T_{\max}$ 

利得行列：

$$
\begin{array}{c|cc}
 & C & D \\
\hline
C & (3, 3) & (0, 8) \\
D & (8, 0) & (1, 1)
\end{array}
$$

---

## 1. TFT vs TFT

しっぺ返し戦略同士の場合、両者とも最初に $C$ を選ぶ。  
その後も相手の $C$ を真似し続けるため、すべての期で $C$ vs $C$ となる。

したがって、各期の利得は $3$ 

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \sum_{t=0}^{T_{\max}-1} 3W^t
= 3 \sum_{t=0}^{T_{\max}-1} W^t
= 3 \cdot \frac{1 - W^{T_{\max}}}{1 - W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \frac{3}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{TFT},\mathrm{TFT}}
= \frac{3}{1-0.8}
= 15
$$

---

## 2. TFT vs All-D

しっぺ返し戦略は最初に $C$ を選び、裏切り戦略は常に $D$ を選ぶ。

したがって、1期目は

$$
C \text{ vs } D
$$



このとき、TFT 側の利得は $0$、All-D 側の利得は $8$ 

2期目以降、TFT は相手の前回行動 $D$ を真似するため、両者とも $D$ を選ぶ。  
したがって、2期目以降はすべて $D$ vs $D$ となり、各期の利得は $1$ 

### TFT 側の割引利得

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= 0 + \sum_{t=1}^{T_{\max}-1} 1 \cdot W^t
= \sum_{t=1}^{T_{\max}-1} W^t
= \frac{W(1-W^{T_{\max}-1})}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= \frac{W}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{TFT},\mathrm{All-D}}
= \frac{0.8}{1-0.8}
= 4
$$

### All-D 側の割引利得

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \sum_{t=1}^{T_{\max}-1} 1 \cdot W^t
= 8 + \sum_{t=1}^{T_{\max}-1} W^t
= 8 + \frac{W(1-W^{T_{\max}-1})}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \frac{W}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{All-D},\mathrm{TFT}}
= 8 + \frac{0.8}{1-0.8}
= 12
$$

---

## 3. All-D vs All-D

裏切り戦略同士の場合、両者とも常に $D$ を選ぶ。

したがって、すべての期で $D$ vs $D$ となり、各期の利得は $1$ 

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \sum_{t=0}^{T_{\max}-1} 1 \cdot W^t
= \sum_{t=0}^{T_{\max}-1} W^t
= \frac{1 - W^{T_{\max}}}{1-W}
$$

無限回繰り返しの場合、

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \frac{1}{1-W}
$$

$W=0.8$ の場合、

$$
V_{\mathrm{All-D},\mathrm{All-D}}
= \frac{1}{1-0.8}
= 5
$$

---

## 4. 利得表

$$
\begin{array}{c|cc}
 & \mathrm{TFT} & \mathrm{All-D} \\
\hline
\mathrm{TFT}
& (15, 15)
& (4, 12) \\
\mathrm{All-D}
& (12, 4)
& (5, 5)
\end{array}
$$


## 得点依存増殖率モデルと得点依存生存率モデル

- 得点依存増殖率モデル
ランダムなプレイヤーが死亡 & 新たに生まれるプレイヤーの戦略は利得に依存するモデル

- 得点依存生存率モデル
利得に依存してプレイヤーが死亡 & 新たに生まれるプレイヤーの戦略はランダムなモデル

In [101]:
class Game:
    def __init__(self, strgId, payMat, N_grid, w):
        self.strgId = strgId # 戦略ID
        self.payMat = payMat # 利得行列
        self.agent_list = [Player(strgId) for _ in range(N_grid)] # プレイヤーのリスト


D = 1 # 格子の次元
N = 100 # 格子点
w = 0.4 # 反復確率